# Imports

In [2]:
import pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

# Pegando os dados

In [3]:
df_escalonado = pd.read_csv("data/base_escalonada.csv")
TARGET = "Close"
DATE_COLUMN = "Date"

In [4]:
# pensando em 3 dias para prever
h = int(len(df_escalonado) * 0.25)
train = df_escalonado[TARGET].iloc[:-h]

test  = df_escalonado[TARGET].iloc[-h:]

# Buscando o melhor RandomForest

In [6]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

# Cria variáveis defasadas para evitar vazamento temporal
dados = df_escalonado.copy()

colunas_features = [
    coluna for coluna in dados.columns
    if coluna not in [TARGET, DATE_COLUMN]
]

X = dados[colunas_features].shift(1)
X[f"{TARGET}_lag_1"] = dados[TARGET].shift(1)
X[f"{TARGET}_lag_2"] = dados[TARGET].shift(2)
X[f"{TARGET}_lag_3"] = dados[TARGET].shift(3)

y = dados[TARGET]

base_modelo = pd.concat([X, y], axis=1).dropna()
X = base_modelo.drop(columns=[TARGET])
y = base_modelo[TARGET]

h_modelo = int(len(X) * 0.25)
X_train = X.iloc[:-h_modelo]
X_test = X.iloc[-h_modelo:]
y_train = y.iloc[:-h_modelo]
y_test = y.iloc[-h_modelo:]

tscv = TimeSeriesSplit(n_splits=5)

def objetivo(trial):
    modelo = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 800),
        max_depth=trial.suggest_int("max_depth", 3, 30),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_categorical(
            "max_features", ["sqrt", "log2", 1.0]
        ),
        bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
        random_state=42,
        n_jobs=-1
    )

    erros = []

    for treino_idx, validacao_idx in tscv.split(X_train):
        modelo.fit(
            X_train.iloc[treino_idx],
            y_train.iloc[treino_idx]
        )

        previsoes = modelo.predict(X_train.iloc[validacao_idx])
        erros.append(
            mean_absolute_error(
                y_train.iloc[validacao_idx],
                previsoes
            )
        )

    return np.mean(erros)

estudo = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

estudo.optimize(objetivo, n_trials=50, show_progress_bar=True)

melhor_modelo = RandomForestRegressor(
    **estudo.best_params,
    random_state=42,
    n_jobs=-1
)

melhor_modelo.fit(X_train, y_train)

previsoes_teste = melhor_modelo.predict(X_test)
mae_teste = mean_absolute_error(y_test, previsoes_teste)

print("Melhores parâmetros:")
print(estudo.best_params)
print(f"MAE médio na validação: {estudo.best_value:.6f}")
print(f"MAE no teste: {mae_teste:.6f}")

[I 2026-09-29 10:12:50,041] A new study created in memory with name: no-name-bb76a1f2-a1c5-4a2d-bfa4-5c4ed2bb78a3
Best trial: 0. Best value: 0.303201:   2%|▏         | 1/50 [00:10<08:34, 10.50s/it]

[I 2026-09-29 10:13:00,625] Trial 0 finished with value: 0.3032011878101817 and parameters: {'n_estimators': 362, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.3032011878101817.


Best trial: 1. Best value: 0.297908:   4%|▍         | 2/50 [00:21<08:24, 10.52s/it]

[I 2026-09-29 10:13:11,176] Trial 1 finished with value: 0.2979083872637446 and parameters: {'n_estimators': 596, 'max_depth': 3, 'min_samples_split': 20, 'min_samples_leaf': 9, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 1 with value: 0.2979083872637446.


Best trial: 1. Best value: 0.297908:   6%|▌         | 3/50 [00:31<08:03, 10.29s/it]

...

[I 2026-09-29 10:17:31,382] Trial 47 finished with value: 0.29762487032020946 and parameters: {'n_estimators': 153, 'max_depth': 3, 'min_samples_split': 15, 'min_samples_leaf': 3, 'max_features': 1.0, 'bootstrap': True}. Best is trial 28 with value: 0.29703065086415.


Best trial: 28. Best value: 0.297031:  98%|█████████▊| 49/50 [04:44<00:02,  2.94s/it]

[I 2026-09-29 10:17:34,592] Trial 48 finished with value: 0.3002207114076689 and parameters: {'n_estimators': 214, 'max_depth': 8, 'min_samples_split': 19, 'min_samples_leaf': 4, 'max_features': 1.0, 'bootstrap': True}. Best is trial 28 with value: 0.29703065086415.


Best trial: 28. Best value: 0.297031: 100%|██████████| 50/50 [04:49<00:00,  5.78s/it]


[I 2026-09-29 10:17:39,349] Trial 49 finished with value: 0.29833474465674525 and parameters: {'n_estimators': 282, 'max_depth': 5, 'min_samples_split': 16, 'min_samples_leaf': 4, 'max_features': 1.0, 'bootstrap': True}. Best is trial 28 with value: 0.29703065086415.
Melhores parâmetros:
{'n_estimators': 140, 'max_depth': 3, 'min_samples_split': 18, 'min_samples_leaf': 8, 'max_features': 1.0, 'bootstrap': True}
MAE médio na validação: 0.297031
MAE no teste: 0.497061


In [ ]:
melhores_params = {'n_estimators': 140, 'max_depth': 3, 'min_samples_split': 18, 'min_samples_leaf': 8, 'max_features': 1.0, 'bootstrap': True}
MAE_FINAL = 0.497061